In [1]:
import pandas as pd
from fpdf import FPDF
from base_utils import read_model_configs, create_all_agent_response_dataframe, find_correct_mc, find_min_max_scaled_error, prepare_response_dataframe, prepare_original_raw_response_dataframe

In [2]:
# all_response_dfs = []
# test_types = ['ggr', 'vlat', 'calvi-trick', 'calvi-standard', 'holf', 'holf2', 'chartqa-test-continuous-human']
# for test_type in test_types:
#     model_response = prepare_response_dataframe(test_type, relax_acc=True, relax_acc_alpha=0.05)
#     model_response['test_type'] = test_type
#     all_response_dfs.append(model_response)

# all_responses_df = pd.concat(all_response_dfs).replace({"calvi-trick": "calvi", "calvi-standard": "calvi"})

In [3]:
all_response_dfs = []
test_types = ['ggr', 'vlat', 'calvi-trick', 'calvi-standard', 'holf', 'holf2', 'chartqa-test-continuous-human']
for test_type in test_types:
    model_response = prepare_response_dataframe(test_type, relax_acc=True, relax_acc_alpha=0.05, dropna=True)
    model_response['test_type'] = test_type
    all_response_dfs.append(model_response)

all_responses_df = pd.concat(all_response_dfs).replace({"calvi-trick": "calvi", "calvi-standard": "calvi"})

13 770 0
53 3108 0
45 2067 0
15 488 0
384 18156 18156
216 10277 0
125 1568 1568


In [31]:
all_raw_responses_df = []
test_types = ['ggr', 'vlat', 'holf', 'calvi-trick', 'calvi-standard', 'holf2', 'chartqa-test-continuous-human']
for test_type in test_types:
    print(test_type)
    model_response = prepare_original_raw_response_dataframe(test_type)
    model_response['test_type'] = test_type
    all_raw_responses_df.append(model_response)

all_raw_responses_df = pd.concat(all_raw_responses_df)
all_raw_responses_df = all_raw_responses_df.replace({"calvi-trick": "calvi", "calvi-standard": "calvi"})

ggr
vlat
holf
calvi-trick
calvi-standard
holf2
chartqa-test-continuous-human


In [122]:
test_types = ['ggr', 'vlat', 'holf', 'calvi-trick', 'calvi-standard', 'holf2', 'chartqa-test-continuous-human']
item_df = []
for test in test_types:
    idf = pd.read_csv(f'https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test}/questions.csv')
    idf['test_type'] = test
    item_df.append(idf)

item_df = pd.concat(item_df).replace({"calvi-trick": "calvi", "calvi-standard": "calvi"})
item_df['item_id'] = item_df['question'] + " ++ " + item_df['image_file']

In [123]:
def combine_mc_str(row):
    choice_str = ""
    for i in range(1,5):
        mc_option = row[f"mc{i}"]
        if mc_option == "write your answer" or mc_option == "Skip":
            continue
        if pd.isnull(mc_option):
            continue

        if i == 1:
            choice_str += "Choices: "
        choice_str += str(mc_option) + ", "

    return choice_str

item_df['choices'] = item_df.apply(combine_mc_str, axis=1)

In [198]:
model_configs = {
    "Salesforce/blip2-flan-t5-xl": {'top_p': 'p06', 'temp': 't1', 'color': '#5ba3cf', 'accent_color': '#8cbedd'},
    "Salesforce/blip2-flan-t5-xxl": {'top_p': 'p1', 'temp': 't10', 'color': '#4c78a8', 'accent_color': '#81a0c2'},
    "llava-hf/llava-1.5-7b-hf": {'top_p': 'p04', 'temp': 't1', 'color': '#f9b574', 'accent_color': '#facb9d'},
    "llava-hf/llava-1.5-13b-hf": {'top_p': 'p1', 'temp': 't04', 'color': '#f58518', 'accent_color': '#f79d46'},
    "liuhaotian/llava-v1.6-34b": {'top_p': 'p1', 'temp': 't04', 'color': '#BF9000', 'accent_color': '#F1C232'},
    "google/pix2struct-chartqa-base": {'top_p': 'p08', 'temp': 't1', 'color': '#ba94c4', 'accent_color': '#c7b8d9'},
    "google/matcha-chartqa": {'top_p': 'p04', 'temp': 't1', 'color': '#8b6db2', 'accent_color': '#a28ac1'},
    "GPT-4V": {'top_p': 'p1', 'temp': 't02', 'color': '#c93739', 'accent_color': '#C6765E'},
    "Human": {'top_p': 'pna', 'temp': 'tna', 'color': '#2e693b', 'accent_color': '#73D287'}
}

In [308]:
# { c: model_configs[c]['color'] for c in [
#     'Human',
#     'GPT-4V',
#     'google/matcha-chartqa',
#     'google/pix2struct-chartqa-base',
#     'liuhaotian/llava-v1.6-34b',
#     'llava-hf/llava-1.5-13b-hf',
#     'llava-hf/llava-1.5-7b-hf',
#     'Salesforce/blip2-flan-t5-xxl',
#     'Salesforce/blip2-flan-t5-xl'
# ]}

In [181]:
# desired_order = [
#     'Human',
#     'GPT-4V',
#     'google/matcha-chartqa',
#     'google/pix2struct-chartqa-base',
#     'liuhaotian/llava-v1.6-34b',
#     'llava-hf/llava-1.5-13b-hf',
#     'llava-hf/llava-1.5-7b-hf',
#     'Salesforce/blip2-flan-t5-xxl',
#     'Salesforce/blip2-flan-t5-xl'
# ]
# agg_df['agent_type'] = agg_df['agent_type'].astype(str)
# agg_df['agent_type'] = pd.Categorical(agg_df['agent_type'], categories=desired_order, ordered=True)
# sorted_df = agg_df.sort_values('agent_type')

def create_aggregate_raw_responses(df):
    agg_df = df.copy()
    agg_df['item_id'] = agg_df['question'] + " ++ " + agg_df['image_file']
    agg_answers = agg_df['agent_response'].fillna("NA")
    
    agg_df = agg_df.groupby(['item_id', 'agent_type']).agg(
        combined_responses=(
            'agent_response', 
            lambda x: '; '.join([f'"{response}" ({count})' for response, count in pd.Series(x).value_counts().items()])
        )
    ).reset_index().sort_values(by="agent_type")

    # agg_df['combined_responses'] = agg_df['agent_type'] + ": " + agg_df['combined_responses'] 

    return agg_df

raw_response_str_df = create_aggregate_raw_responses(all_raw_responses_df)

In [182]:
# raw_response_str_df

In [299]:
def create_aggregate_responses(df):
    agg_df = df.copy()
    agg_df['item_id'] = agg_df['question'] + " ++ " + agg_df['image_file']
    agg_answers = agg_df['agent_response'].fillna("NA")
    
    agg_df = agg_df.groupby(['item_id', 'agent_type']).agg(
        is_correct_mean=('is_correct', 'mean'),
        combined_responses=(
            'agent_response', 
            lambda x: '; '.join([f"{response} ({count})" for response, count in pd.Series(x).value_counts().items()])
        )
    ).reset_index()
    agg_df['is_correct_mean_str'] = agg_df['is_correct_mean'].apply(lambda x : round(x*100, 1)).astype(str) + "%"
    
    return agg_df[['item_id', 'agent_type', 'is_correct_mean', 'is_correct_mean_str']]

processed_response_str_df = create_aggregate_responses(all_responses_df)
print(len(processed_response_str_df))
raw_response_str_df = create_aggregate_raw_responses(all_raw_responses_df)
print(len(raw_response_str_df))
agg_df = pd.merge(processed_response_str_df, raw_response_str_df, on=["item_id", "agent_type"], how='right')

next_line_token = ' <br /> '
agg_df['is_correct_mean_str'] = agg_df['is_correct_mean_str'].fillna('N/A')
agg_df['styled_agent'] = agg_df['agent_type'].apply(lambda m : f'<b style="color:{model_configs[m]["color"]};"> {m} </b>')
agg_df['combined_responses'] = agg_df.apply(
    lambda r : f'<span class="{r["agent_type"].replace("/", "-").replace(".", "-")}" > {r["styled_agent"]} -- <b>Accuracy</b>: {r["is_correct_mean_str"]}; <b>Responses</b>: {r["combined_responses"]}</span>',
    axis=1                                 
)

# agg_df + " -- <b>Accuracy</b>: " + agg_df['is_correct_mean_str'] + "; <b>Responses</b>: " + agg_df['combined_responses']

agg_df = agg_df.groupby(['item_id']).agg(
    combined_responses=(
        'combined_responses', 
        lambda x: next_line_token.join(x)
    )
).reset_index()

6625
7307


In [300]:
def rename_models(r):
    if r != "Human":
        return "Model"
    else:
        return r

def get_image_link(r):
    test_type = r['test_type']
    image_file = r['image_file']
    if test_type == "holf":
        partial = image_file.split("_")[0]
        q = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{partial}/{image_file}"
    else:
        q = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{image_file}"
    return q

def invalid_included_aggregation(group):
    if 'mod_agent_type' in group.columns:
        agent_type = group['mod_agent_type'].iloc[0]
    else:
        agent_type = group['agent_type'].iloc[0]
    
    if agent_type != 'Human': 
        # group['is_correct'].sum() / 10
        return group['is_correct'].sum() / 10
    else:
        return group['is_correct'].mean()


all_responses_df['mod_agent_type'] = all_responses_df['agent_type'].apply(rename_models)
# agent_responses = all_responses_df.groupby(['mod_agent_type', 'question', 'image_file', 'test_type', 'correct_answer'])['is_correct'].mean().reset_index()
# averaged_responses = all_responses_df.groupby(['agent_type', 'question', 'image_file', 'test_type', 'correct_answer'])['is_correct'].mean().reset_index()
agent_responses = all_responses_df.groupby(['mod_agent_type', 'question', 'image_file', 'test_type', 'correct_answer'])[all_responses_df.columns].apply(invalid_included_aggregation).reset_index()
agent_responses = agent_responses.rename(columns={0: 'is_correct'})
averaged_responses = all_responses_df.groupby(['agent_type', 'question', 'image_file', 'test_type', 'correct_answer'])[all_responses_df.columns].apply(invalid_included_aggregation).reset_index()
averaged_responses = averaged_responses.rename(columns={0: 'is_correct'})

agent_responses = agent_responses.pivot(index=['question', 'image_file', 'test_type', 'correct_answer'], columns="mod_agent_type", values='is_correct').reset_index()
agent_responses['agent_diff'] = agent_responses['Human'] - agent_responses['Model']
agent_responses['agent_sum'] = agent_responses['Human'] + agent_responses['Model']
agent_responses['image_link'] = agent_responses.apply(get_image_link, axis=1)
agent_responses['item_id'] = agent_responses['question'] + " ++ " + agent_responses['image_file']

In [301]:
# agg_combined_df

In [302]:
agg_combined_df = pd.merge(agg_df, agent_responses, on="item_id")
agg_combined_df = pd.merge(agg_combined_df, item_df[['item_id', 'choices']], on='item_id')
agg_combined_df['item_str'] = agg_combined_df.apply(
    lambda r : f'Question: "{r["question"]}" {r["choices"]} Correct Answer: "{r["correct_answer"]}"',
    axis=1
)

In [303]:
def fix_image_link(row):
    test_type = row['test_type']
    image_file = row['image_file']
    image_link = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{image_file}"

    if test_type == "holf":
        partial = image_file.split("_")[0]
        image_link = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{partial}/{image_file}"
    
    return image_link
    
agg_combined_df['image_link'] = agg_combined_df.apply(fix_image_link, axis=1)

In [304]:
agg_combined_df.to_json("./agg_combined_df.json", orient='records')

In [161]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak


def create_pdf(df, pdf_output_path = "./output.pdf"):
    
    pdf = SimpleDocTemplate(pdf_output_path, pagesize=A4)
    
    # Get default stylesheet and customize it
    styles = getSampleStyleSheet()
    bold_style = ParagraphStyle(name='Bold', parent=styles['Normal'], fontSize=12, fontName="Helvetica-Bold")
    normal_style = ParagraphStyle(name='Normal', parent=styles['Normal'], fontSize=12, fontName="Helvetica")
    line_style = ParagraphStyle(name='Line', parent=styles['Normal'], fontSize=12, fontName="Helvetica-Bold", textColor='#000000')
    
    content = []
    for idx, row in df.iterrows():
        content.append(Paragraph(row["item_str"], normal_style))
    
        content.append(Spacer(1, 0.2 * inch))
        test_type = row['test_type']
        image_file = row['image_file']
        image_link = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{image_file}"

        if test_type == "holf":
            partial = image_file.split("_")[0]
            image_link = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{partial}/{image_file}"
        
        content.append(Image(image_link, width=6*inch, height=4*inch))

        content.append(Paragraph("---------------------------------", line_style))

        for line in row['combined_responses'].split(next_line_token):
            content.append(Paragraph(line, normal_style))
    
        # Add a page break after each question
        content.append(PageBreak())
    print("DONE")

    pdf.build(content)

create_pdf(agg_combined_df)

DONE


In [173]:
# agg_combined_df

In [174]:
agg_combined_df.to_json("./agg_combined_df.json", orient='records')

In [163]:
# for test in ['vlat', 'ggr', 'holf', 'calvi', 'chartqa-test-continuous-human', 'holf2']:
    

In [98]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak


def model_average_string(question, image_file, averaged_responses):
    averaged_model_responses = averaged_responses[
        (averaged_responses["question"] == question) &
        (averaged_responses["image_file"] == image_file)
    ]
    final_str = []
    for idx, row in averaged_model_responses.iterrows():
        avg_accuracy = round(row["is_correct"], 2)
        agent_type = row['agent_type']
        final_str.append(f"{agent_type}: {avg_accuracy} ")
    return final_str
        

def create_pdf(filtered_df, all_raw_responses_df, averaged_responses, pdf_output_path = "./output_calvi.pdf"):
    
    pdf = SimpleDocTemplate(pdf_output_path, pagesize=A4)
    
    # Get default stylesheet and customize it
    styles = getSampleStyleSheet()
    bold_style = ParagraphStyle(name='Bold', parent=styles['Normal'], fontSize=12, fontName="Helvetica-Bold")
    normal_style = ParagraphStyle(name='Normal', parent=styles['Normal'], fontSize=12)
    line_style = ParagraphStyle(name='Line', parent=styles['Normal'], fontSize=12, fontName="Helvetica-Bold", textColor='#000000')
    
    # List to hold content elements
    content = []
    
    # Loop through each row in the filtered DataFrame
    for idx, row in filtered_df.iterrows():
        # Add the question text with bold style
        content.append(Paragraph(f"Question: {row['question']} (Image File: {row['image_file']})", bold_style))
    
        # Add correct answer and other details
        content.append(Paragraph(f"Correct Answer: {row['correct_answer']}", normal_style))
        content.append(Paragraph(f"Avg. prop. correct for Humansxx: {round(row['Human'], 2)}; across all Models: {round(row['Model'], 2)}", normal_style))

        for avg_str in model_average_string(row['question'], row['image_file'], averaged_responses):
            content.append(Paragraph(avg_str, normal_style))
        
        # content.append(Paragraph("GPT-4V:", bold_style))
    
        # # Add agent responses
        selected_df = all_raw_responses_df[
            (all_raw_responses_df['question'] == row['question']) &
            (all_raw_responses_df['image_file'] == row['image_file']) 
        # & (all_raw_responses_df['agent_type'] == 'GPT-4V')
        ][['agent_response']].value_counts().reset_index()
    
        for i2, row2 in selected_df.iterrows():
            content.append(Paragraph("---------------------------------", line_style))
            content.append(Paragraph(f"{row2['agent_response']} ({row2['count']})", normal_style))
            content.append(Spacer(1, 0.1 * inch))

        content.append(Paragraph("---------------------------------", line_style))
    
        content.append(Spacer(1, 0.2 * inch))

        image_file = row['image_link']
        if test_type == "holf":
            partial = image_file.split("_")[0]
            image_file = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{partial}/{image_file}"
        else:
            image_file = f"https://data-visualization-benchmark.s3.us-west-2.amazonaws.com/{test_type}/images/{image_file}"

        content.append(Image(image_file, width=6*inch, height=4*inch))
    
        # Add a page break after each question
        content.append(PageBreak())
    
    # Build the PDF
    pdf.build(content)

In [128]:
# agent_responses

In [99]:
def create_test_pdf(test_type, agent_responses, all_raw_responses_df, averaged_responses, ascending=True, measure="agent_diff"):
    filtered_df = agent_responses[agent_responses['test_type'] == test_type].drop_duplicates()
    filtered_df = filtered_df.sort_values(by=measure, ascending=ascending)
    ascending_str = "ascending" if ascending else "descending"
    pdf_output_path = f"./invalid_only/{measure}_{ascending_str}/output_{test_type}.pdf"
    create_pdf(filtered_df, all_raw_responses_df, averaged_responses, pdf_output_path)

In [100]:
# create_test_pdf('holf', agent_responses, all_raw_responses_df, averaged_responses)
# averaged_responses['agent_type'].unique()
# all_responses_df['agent_type'].unique()

In [129]:
# for t in ['ggr', 'vlat', 'calvi', ]: #'holf', 'holf2', 'chartqa-test-continuous-human'
#     create_test_pdf(t, agent_responses, all_raw_responses_df, averaged_responses, ascending=False, measure="agent_diff")

# for t in ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']:
#     create_test_pdf(t, agent_responses, all_raw_responses_df, averaged_responses, ascending=True, measure="agent_diff")

# for t in ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']:
#     create_test_pdf(t, agent_responses, all_raw_responses_df, averaged_responses, ascending=False, measure="agent_sum")

# for t in ['ggr', 'vlat', 'calvi', 'holf', 'holf2', 'chartqa-test-continuous-human']:
#     create_test_pdf(t, agent_responses, all_raw_responses_df, averaged_responses, ascending=True, measure="agent_sum")